##  导入数据

In [ ]:
import pandas as pd
df = pd.read_csv('combined_data.csv')
df.head()

In [ ]:
train_data=df[['frequency','Time gap','pur_in_degree','sum_length','new_length','eigenvector_centrality','clustering_coefficient','degree_centrality','betweenness_centrality','closeness','construct_label','dynamic_data']]
train_data.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 绘制相关性热力图
plt.subplots(figsize=(8, 8))  # 设置画面大小 
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号 
sns.heatmap(train_data.corr(), annot=True, vmax=1, square=True, cmap="Blues") 
plt.title('相关性热力图')
plt.show()

In [ ]:
train_data[::-1].describe().loc[['min','max','mean','std','50%']].T

## 更改一

In [ ]:
# 对类别数据进行独热编码
df_encoded = pd.get_dummies(train_data, columns=['dynamic_data'])
df_encoded

In [ ]:
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

df_encoded[['construct_label']].boxplot()  #对数据框中每列画箱线图，pandas自己有处理的过程，很方便
plt.show()

In [ ]:
from scipy.stats.mstats import winsorize

df_encoded['construct_label']=winsorize(df_encoded['construct_label'],limits=[0, 0.01])

In [ ]:
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

df_encoded[['construct_label']].boxplot()  #对数据框中每列画箱线图，pandas自己有处理的过程，很方便
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# 计算每一列的熵
def entropy(series):
    p = series / series.sum()
    return -np.sum(p * np.log(p))

In [ ]:
def entropy_data(df):
    entropies = df.apply(entropy)

    # 计算每一列的权重
    weights = 1 - entropies / entropies.sum()

    # 将权重应用于数据并合并为一列
    weighted_sum = (df * weights).sum(axis=1)

    # 添加合并后的列到DataFrame
    df['Merged'] = weighted_sum

    return(df)

In [ ]:
M1=df_encoded[['pur_in_degree','new_length']]
M2=df_encoded[['clustering_coefficient','degree_centrality']]
M3=df_encoded[['closeness','betweenness_centrality','eigenvector_centrality']]

In [ ]:
M1=entropy_data(M1)
M2=entropy_data(M2)
M3=entropy_data(M3)

In [ ]:
after_M1=M1[['Merged']]
after_M2=M2[['Merged']]
after_M3=M3[['Merged']]

In [ ]:
last_data=pd.concat([df_encoded[['frequency','Time gap']],after_M1,after_M2,after_M3,df_encoded[['dynamic_data_增长','dynamic_data_稳定','dynamic_data_衰退']]], axis=1)
last_data.columns=['F','R','M1','M2','M3','T1','T2','T3']
last_data.head()

In [ ]:
# 数据标准化
def zscore_data(data):
    data2=(data-data.mean(axis=0))/data.std(axis=0)
    data2.columns=["Z"+i for i in data.columns]
    return data2
 
z_data = zscore_data(last_data[['F','R','M1','M2','M3']])
z_data =pd.concat([z_data,df_encoded[['dynamic_data_增长','dynamic_data_稳定','dynamic_data_衰退']]], axis=1)
z_data.head()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score,davies_bouldin_score

sse=[]
CH_data=[]
DB_data=[]

for i in range(2,20):
    kmeans = KMeans(n_clusters=i,init='k-means++',max_iter=300,n_init=10,random_state=0)
    result_list =kmeans.fit_predict(z_data)
    
    CH_index=calinski_harabasz_score(z_data,result_list)
    DB_index=davies_bouldin_score(z_data,result_list)
    
    print(f"方差比为: {CH_index}")
    print(f"DB值为: {DB_index}")
    print('SSE值为:',kmeans.inertia_)
    
    sse.append(kmeans.inertia_)
    CH_data.append(CH_index)
    DB_data.append(DB_index)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(2,20),sse)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

In [ ]:
plt.plot(range(2,20),CH_data)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('CH_data')
plt.show()

In [ ]:
CH_data

In [ ]:
plt.plot(range(2,20),DB_data)
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('DB_data')
plt.show()

In [ ]:
DB_data

In [ ]:
from sklearn.cluster import KMeans
kmeans_model = KMeans(n_clusters=5,init='k-means++',max_iter=300,n_init=10,random_state=0)
kmeans_model.fit(z_data)

In [ ]:
z_data

In [ ]:
kmeans_cc=kmeans_model.cluster_centers_   # 聚类中心
kmeans_cc

In [ ]:
temp_data=pd.DataFrame(kmeans_cc)
temp_data.columns=['F','R','M1','M2','M3','T1','T2','T3']

In [ ]:
temp_data

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 归一化函数
def normalize_to_minus_one_to_one(column):
    # 计算最大值和最小值
    max_val = column.max()
    min_val = column.min()
    # 归一化到 [-1, 1] 区间
    normalized_column = (column - min_val) / (max_val - min_val) * 2 - 1
    return normalized_column

# 对每列数据进行归一化
df_normalized = temp_data.apply(normalize_to_minus_one_to_one)

In [ ]:
df_normalized

In [ ]:
columns_order = ['R','M3','M1','M2','F','T3','T1','T2']  # 这里是你想要的新的列顺序

In [ ]:
# 使用 loc 方法重新排列列
df_normalized=df_normalized.loc[:, columns_order]
df_normalized

In [ ]:
from pyecharts.charts import Radar
from pyecharts import options as opts
import numpy as np

#客户价值雷达图


radar = Radar()
radar.add_schema(
                 axisline_opt=opts.LineStyleOpts(is_show=True, color='rgba(238, 197, 102, 1)'),
                textstyle_opts=opts.TextStyleOpts(color='#000000'),
                 schema=[opts.RadarIndicatorItem(name="最近知识共享距离的时间",min_=-2, max_=1),
                         opts.RadarIndicatorItem(name="连通性",min_=-2, max_=1),
                         opts.RadarIndicatorItem(name="创新性",min_=-2, max_=1),
                         opts.RadarIndicatorItem(name='聚集性',min_=-2, max_=1),
                         opts.RadarIndicatorItem(name='知识共享频率',min_=-2, max_=1),
                         opts.RadarIndicatorItem(name="T3",min_=-2, max_=1),
                         opts.RadarIndicatorItem(name="T1",min_=-2, max_=1),
                         opts.RadarIndicatorItem(name="T2",min_=-2, max_=1),
                         ])

radar.add('普通用户-增长型', [list(df_normalized.iloc[0])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#063fe1', width=0.1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#063fe1', opacity=0.1))

radar.add('普通用户-稳定型', [list(df_normalized.iloc[1])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#5cb047', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#5cb047', opacity=0.05))

radar.add('创新中介者', [list(df_normalized.iloc[2])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#FFA500', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#FFA500',opacity=0.05))

radar.add('普通用户-衰退型', [list(df_normalized.iloc[3])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#e10606', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#e10606',opacity=0.05))

radar.add('领导者', [list(df_normalized.iloc[4])], symbol='none',
          label_opts=opts.LabelOpts(is_show=True),
          linestyle_opts=opts.LineStyleOpts(color='#87CEEB', width=1, opacity=0.6),
          areastyle_opts=opts.AreaStyleOpts(color='#87CEEB',opacity=0.05))

radar.set_global_opts(legend_opts=opts.LegendOpts(is_show=True, selected_mode='flase', pos_bottom=5),
                      title_opts=opts.TitleOpts(title="动态模型客户价值雷达图", pos_left='center',
                                                title_textstyle_opts=opts.TextStyleOpts(font_size=20)))


radar.render_notebook()
